In [ ]:
# | default_exp features.fsr_genomewide

In [ ]:
# | export
from __future__ import annotations
import pandas as pd
import structlog
from kreview.eval_engine import FeatureEvaluator

log = structlog.get_logger()

In [ ]:
# | export
class FSREvaluator(FeatureEvaluator):
    """Extracts the short/long fragment size ratio across predefined structural metrics."""

    name = "FsrGenomewide"
    source_file = ".FSR.parquet"
    tier = 1
    category = "fragmentation"

    def extract(self, df: pd.DataFrame) -> dict[str, float]:
        extracted = {}
        try:
            if df.empty:
                return extracted

            cols = set(df.columns)
            target_metrics = [
                "short_long_ratio",
                "short_long_log2",
                "short_frac",
                "long_frac",
            ]

            for m in target_metrics:
                if m in cols:
                    extracted[f"global_{m}"] = float(df[m].median())

            return extracted
        except Exception as e:
            log.exception("extraction_failed", evaluator=self.name, error=str(e))
            return {}

In [ ]:
# | test
evaluator = FSREvaluator()
df_test = pd.DataFrame({"short_long_ratio": [1.5], "short_frac": [0.6]})
res = evaluator.extract(df_test)
assert res["global_short_long_ratio"] == 1.5
assert res["global_short_frac"] == 0.6